# Do more people want out of their gym contract in spring? The notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/john-bruene/pointzerofive-data/blob/main/gym-contract-spring/story_009_gym_contract.ipynb)

This R notebook recomputes every statistic in
[Do more people want out of their gym contract in spring?](https://pointzerofive.net/2026/11/gym-contract-spring),
story #009 on *pointzerofive*, from the two Google Trends exports. Colab opens it with the R
runtime; run it top to bottom (`Runtime → Run all`).

Every number is checked against the published results file with `stopifnot`. Requires base R,
`jsonlite` and `nlme` (both preinstalled in Colab). The pipeline that wrote the dataset is
`gym_pipeline.R`; its console output, with R and package versions, is
`gym-contract-spring.output.txt`.

In [1]:
suppressPackageStartupMessages({ library(jsonlite); library(nlme) })

BASE <- "https://raw.githubusercontent.com/john-bruene/pointzerofive-data/main/gym-contract-spring/"
REPO <- ".."

# A file of this story: next to the notebook (pointzerofive-data), in the website's repository,
# else downloaded from pointzerofive-data.
fetch_path <- function(rel) {
  for (local in c(basename(rel), file.path(REPO, rel))) if (file.exists(local)) return(local)
  paste0(BASE, basename(rel))
}

near <- function(a, b, tol = 1e-6) stopifnot(abs(a - b) <= tol)

raw <- read.csv(fetch_path("notebooks/raw/fitness/trends_monthly.csv"), encoding = "UTF-8",
                stringsAsFactors = FALSE)
published <- fromJSON(fetch_path("data/gym-contract-spring.json"), simplifyVector = FALSE)
print(table(raw$term))


fitnessstudio       vertrag 
          275           275 


## 1. The two series

Google Trends, web search, Germany, monthly; each term exported on its own (index 0 to 100 within
the export). A series starts in the first calendar year in which every month has a value of at
least 1 and ends in October 2026 (November is incomplete). Months with a value of 0 after the start
count as missing: Google shows 0 when too few people searched.

In [2]:
series <- function(id) {
  d <- raw[raw$term == id & raw$month <= "2026-10", ]
  d$year <- as.integer(substr(d$month, 1, 4))
  d$m <- as.integer(substr(d$month, 6, 7))
  full <- tapply(d$value >= 1, d$year, function(x) length(x) == 12 && all(x))
  start <- min(as.integer(names(full)[full]))
  d <- d[d$year >= start, ]
  zeros <- sum(d$value == 0)
  d <- d[d$value > 0, ]
  d$t <- (d$year - 2004L) * 12L + d$m
  d$log_value <- log(d$value)
  attr(d, "start") <- start; attr(d, "zeros") <- zeros
  d
}
s <- list(fitnessstudio = series("fitnessstudio"), vertrag = series("vertrag"))
sm <- published$summary
stopifnot(attr(s$fitnessstudio, "start") == sm$start, nrow(s$fitnessstudio) == sm$months,
          attr(s$vertrag, "start") == sm$vertragStart, nrow(s$vertrag) == sm$vertragMonths,
          attr(s$vertrag, "zeros") == sm$vertragZeros)
for (id in names(s)) cat(sprintf("%s: from %d, %d months, %d zeros left out\n", id,
                                 attr(s[[id]], "start"), nrow(s[[id]]), attr(s[[id]], "zeros")))

fitnessstudio: from 2011, 190 months, 0 zeros left out
vertrag: from 2020, 57 months, 25 zeros left out


## 2. First results (Fig. 1)

Raw means of the complete years 2011 to 2025, spring (February to April) against the other months,
and the month with the most searches in each year.

In [3]:
fs <- s$fitnessstudio
complete <- fs[fs$year <= 2025, ]
spring <- complete$m %in% 2:4
near(round(mean(complete$value[spring]), 2), sm$rawSpring); near(round(mean(complete$value[!spring]), 2), sm$rawOther)
top <- sapply(split(complete, complete$year), function(y) y$m[which.max(y$value)])
stopifnot(sum(top == 1) == sm$januaryPeaks, length(top) == sm$completeYears)
cat(sprintf("spring %.2f, other months %.2f; January the top month in %d of %d years\n",
            mean(complete$value[spring]), mean(complete$value[!spring]), sum(top == 1), length(top)))
print(round(tapply(fs$value, fs$year, mean), 1))

spring 29.91, other months 30.48; January the top month in 8 of 15 years


2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025 2026 
 9.8 14.4 14.3 17.8 22.2 22.5 25.4 27.8 29.5 59.4 58.0 53.6 44.5 33.8 22.2 16.7 


## 3. The test: GLS with AR(1) errors

Log index on a spring dummy and one dummy per year; AR(1) errors over the month number (`corAR1`
uses the differences of `t`, so a missing month is a gap); REML. Effect = 100 · (exp(b) − 1);
95% interval b ± t(0.975, df) · se with df = N − p, converted the same way.

In [4]:
windows <- list(feb_apr = 2:4, jan_mar = 1:3, mar_may = 3:5)
path_data <- function(window, term, years) {
  d <- s[[term]]
  if (years == "without_covid") d <- d[!d$year %in% c(2020, 2021), ]
  if (years == "since_2023") d <- d[d$year >= 2023, ]
  d$spring <- as.integer(d$m %in% windows[[window]])
  d
}
test_path <- function(d) {
  d$year_f <- factor(d$year)
  fit <- gls(log_value ~ spring + year_f, data = d, correlation = corAR1(form = ~ t), method = "REML")
  tt <- summary(fit)$tTable
  b <- tt["spring", "Value"]; se <- tt["spring", "Std.Error"]
  df <- fit$dims$N - fit$dims$p; q <- qt(0.975, df)
  pct <- function(x) 100 * (exp(x) - 1)
  list(effect = pct(b), ci_low = pct(b - q * se), ci_high = pct(b + q * se), p = tt["spring", "p-value"],
       n = nrow(d), b = b, se = se, fit = fit)
}
main <- test_path(path_data("feb_apr", "fitnessstudio", "all"))
pub <- published$specs[[1]]
stopifnot(identical(unlist(pub$choices), c(window = "feb_apr", term = "fitnessstudio", years = "all")))
near(main$effect, pub$effect); near(main$ci_low, pub$ci_low); near(main$ci_high, pub$ci_high)
near(main$p, pub$p, 1e-9)
cat(sprintf("spring %+.1f per cent, 95%% CI [%+.1f, %+.1f], p = %.4f, n = %d\n", main$effect,
            main$ci_low, main$ci_high, main$p, main$n))
print(summary(main$fit)$tTable[1:2, ])
near(published$estimation$rows[[2]]$values[[1]], main$b)

spring -6.9 per cent, 95% CI [-12.9, -0.5], p = 0.0365, n = 190


                  Value  Std.Error   t-value      p-value
(Intercept)  2.27712225 0.07506456 30.335519 3.640831e-71
spring      -0.07158203 0.03396356 -2.107613 3.650558e-02


## 4. The months of the year (Fig. 2)

For each complete year 2011 to 2025, each month's deviation from its year's mean on the log scale,
averaged over the years, with a 95% t-interval, as per cent. Descriptive only.

In [5]:
dev <- complete$log_value - ave(complete$log_value, complete$year)
prof <- t(sapply(1:12, function(m) {
  x <- dev[complete$m == m]; ci <- t.test(x)$conf.int
  c(month = m, pct = 100 * (exp(mean(x)) - 1), lo = 100 * (exp(ci[1]) - 1), hi = 100 * (exp(ci[2]) - 1))
}))
for (m in 1:12) {
  p <- published$profile[[m]]
  near(round(prof[m, "pct"], 2), p$pct); near(round(prof[m, "lo"], 2), p$lo); near(round(prof[m, "hi"], 2), p$hi)
}
print(round(prof, 1))

      month  pct    lo   hi
 [1,]     1 16.9   1.3 35.1
 [2,]     2 -3.9 -13.5  6.8
 [3,]     3 -4.1 -12.5  5.1
 [4,]     4 -2.4  -9.4  5.2
 [5,]     5 -2.3  -8.2  4.0
 [6,]     6 -5.9 -11.6  0.1
 [7,]     7 -4.6 -12.3  3.9
 [8,]     8  4.7  -2.9 12.8
 [9,]     9  4.0  -4.9 13.6
[10,]    10  9.8  -0.5 21.0
[11,]    11 -1.3  -8.5  6.5
[12,]    12 -8.1 -15.4 -0.2


## 5. All 18 calculation paths

Spring: February to April, January to March or March to May. Search term: "Fitnessstudio kündigen"
or "Fitnessvertrag kündigen". Years: all, without 2020 and 2021, or only 2023 to 2026.

In [6]:
grid <- expand.grid(years = c("all", "without_covid", "since_2023"), term = c("fitnessstudio", "vertrag"),
                    window = c("feb_apr", "jan_mar", "mar_may"), stringsAsFactors = FALSE)[, 3:1]
out <- list()
for (i in seq_len(nrow(grid))) {
  ch <- grid[i, ]
  r <- test_path(path_data(ch$window, ch$term, ch$years))
  p <- published$specs[[i]]
  stopifnot(identical(unname(unlist(p$choices)), unname(unlist(ch))), r$n == p$n)
  near(r$effect, p$effect); near(r$ci_low, p$ci_low); near(r$ci_high, p$ci_high); near(r$p, p$p, 1e-9)
  out[[i]] <- data.frame(ch, n = r$n, effect = round(r$effect, 1), ci_low = round(r$ci_low, 1),
                         ci_high = round(r$ci_high, 1), p = round(r$p, 4), yes = r$p < 0.05 && r$effect > 0)
}
specs <- do.call(rbind, out)
print(specs, row.names = FALSE)
cat(sprintf("yes on %d of %d paths\n", sum(specs$yes), nrow(specs)))

  window          term         years   n effect ci_low ci_high      p   yes
 feb_apr fitnessstudio           all 190   -6.9  -12.9    -0.5 0.0365 FALSE
 feb_apr fitnessstudio without_covid 166   -6.3  -11.8    -0.4 0.0363 FALSE
 feb_apr fitnessstudio    since_2023  46   14.5    3.6    26.5 0.0091  TRUE
 feb_apr       vertrag           all  57   -0.5  -20.4    24.4 0.9657 FALSE
 feb_apr       vertrag without_covid  33   15.5  -12.1    51.8 0.2878 FALSE
 feb_apr       vertrag    since_2023  21   15.3   -6.0    41.4 0.1586 FALSE
 jan_mar fitnessstudio           all 190    9.1    1.9    16.7 0.0123  TRUE
 jan_mar fitnessstudio without_covid 166    8.5    2.0    15.4 0.0097  TRUE
 jan_mar fitnessstudio    since_2023  46   28.7   20.4    37.6 0.0000  TRUE
 jan_mar       vertrag           all  57   13.8   -7.4    39.9 0.2123 FALSE
 jan_mar       vertrag without_covid  33   27.1    1.9    58.4 0.0342  TRUE
 jan_mar       vertrag    since_2023  21   14.6   -4.0    36.7 0.1227 FALSE
 mar_may fit

yes on 5 of 18 paths


## Data and methods

- **Search interest:** [Google Trends](https://trends.google.com), web search, Germany, monthly;
  "Fitnessstudio kündigen" and "Fitnessvertrag kündigen", each exported on its own by John Brüne on
  18 November 2026. "Source: Google Trends". Google samples its data; a new export can differ by a
  few points.
- **Rules:** a series starts in the first year with a value in every month and ends in October
  2026; zeros after the start count as missing (rule set after the pitch, affects only
  "Fitnessvertrag kündigen").
- **Test:** GLS with AR(1) errors, spring dummy and year dummies, t-test of the spring coefficient,
  α = 0.05, two-sided; 95% confidence interval from the same t distribution. No random numbers.
- **Pre-registration:** question, terms, period, year rule, forks, test and the smallest relevant
  effect (10 per cent) were fixed in the pitch (`pitch.md`) before the two exports were opened.

Notebook and code: © John Brüne, all rights reserved.